# Installing Whisper

The commands below will install the Python packages needed to use Whisper models and evaluate the transcription results.

In [1]:
! pip install git+https://github.com/openai/whisper.git
! pip install jiwer

  Cloning https://github.com/openai/whisper.git to /tmp/nix-shell.Ag0J9A/pip-req-build-kx61ap7r
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/nix-shell.Ag0J9A/pip-req-build-kx61ap7r
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Loading the LibriSpeech dataset

The following will load the test-clean split of the LibriSpeech corpus using torchaudio.

In [ ]:
import os
import numpy as np
import torch
import pandas as pd
import whisper
import torchaudio

from tqdm.notebook import tqdm


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/home/chiedozie/projects/ADLS/hush/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
class LibriSpeech(torch.utils.data.Dataset):
    """
    A simple class to wrap LibriSpeech and trim/pad the audio to 30 seconds.
    It will drop the last few seconds of a very small portion of the utterances.
    """
    def __init__(self, split="test-clean", device=DEVICE):
        self.dataset = torchaudio.datasets.LIBRISPEECH(
            root=os.path.expanduser("~/.cache"),
            url=split,
            download=True,
        )
        self.device = device

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, item):
        audio, sample_rate, text, _, _, _ = self.dataset[item]
        assert sample_rate == 16000
        audio = whisper.pad_or_trim(audio.flatten()).to(self.device)
        mel = whisper.log_mel_spectrogram(audio)
        
        return (mel, text)

In [35]:
dataset = LibriSpeech("test-clean")
dataset = torch.utils.data.Subset(dataset, range(100))
loader = torch.utils.data.DataLoader(dataset, batch_size=16)

# Running inference on the dataset using a base Whisper model

The following will take a few minutes to transcribe all utterances in the dataset.

In [36]:
model = whisper.load_model("base.en")
print(
    f"Model is {'multilingual' if model.is_multilingual else 'English-only'} "
    f"and has {sum(np.prod(p.shape) for p in model.parameters()):,} parameters."
)

Model is English-only and has 71,825,408 parameters.


In [37]:
# predict without timestamps for short-form transcription
options = whisper.DecodingOptions(language="en", without_timestamps=True)

In [38]:
hypotheses = []
references = []

for mels, texts in tqdm(loader):
    results = model.decode(mels, options)
    hypotheses.extend([result.text for result in results])
    references.extend(texts)

  0%|          | 0/7 [00:00<?, ?it/s]

In [39]:
data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references))
data

,hypothesis,reference
0,"He hoped there would be stew for dinner, turni...",HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...
1,"Stuffered into you, his belly counseled him.",STUFF IT INTO YOU HIS BELLY COUNSELLED HIM
2,After early nightfall the yellow lamps would l...,AFTER EARLY NIGHTFALL THE YELLOW LAMPS WOULD L...
3,"Hello Bertie, any good in your mind?",HELLO BERTIE ANY GOOD IN YOUR MIND
4,Number 10. Fresh Nelly is waiting on you. Good...,NUMBER TEN FRESH NELLY IS WAITING ON YOU GOOD ...
...,...,...
95,"There's one and there's another, the Dudley an...",THERE'S ONE AND THERE'S ANOTHER THE DUDLEY AND...
96,It is only a pencil outline by Edward Byrne Jo...,IT IS ONLY A PENCIL OUTLINE BY EDWARD BURNE JO...
97,"Every plant in the grass is set formally, grow...",EVERY PLANT IN THE GRASS IS SET FORMALLY GROWS...
98,Exquisite order and universal with eternal lif...,EXQUISITE ORDER AND UNIVERSAL WITH ETERNAL LIF...


# Calculating the word error rate

Now, we use our English normalizer implementation to standardize the transcription and calculate the WER.

In [40]:
import jiwer
from whisper.normalizers import EnglishTextNormalizer

normalizer = EnglishTextNormalizer()

In [41]:
data["hypothesis_clean"] = [normalizer(text) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data

,hypothesis,reference,hypothesis_clean,reference_clean
0,"He hoped there would be stew for dinner, turni...",HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...,he hoped there would be stew for dinner turnip...,he hoped there would be stew for dinner turnip...
1,"Stuffered into you, his belly counseled him.",STUFF IT INTO YOU HIS BELLY COUNSELLED HIM,stuffered into you his belly counseled him,stuff it into you his belly counseled him
2,After early nightfall the yellow lamps would l...,AFTER EARLY NIGHTFALL THE YELLOW LAMPS WOULD L...,after early nightfall the yellow lamps would l...,after early nightfall the yellow lamps would l...
3,"Hello Bertie, any good in your mind?",HELLO BERTIE ANY GOOD IN YOUR MIND,hello bertie any good in your mind,hello bertie any good in your mind
4,Number 10. Fresh Nelly is waiting on you. Good...,NUMBER TEN FRESH NELLY IS WAITING ON YOU GOOD ...,number 10 fresh nelly is waiting on you good n...,number 10 fresh nelly is waiting on you good n...
...,...,...,...,...
95,"There's one and there's another, the Dudley an...",THERE'S ONE AND THERE'S ANOTHER THE DUDLEY AND...,there is one and there is another the dudley a...,there is one and there is another the dudley a...
96,It is only a pencil outline by Edward Byrne Jo...,IT IS ONLY A PENCIL OUTLINE BY EDWARD BURNE JO...,it is only a pencil outline by edward byrne jo...,it is only a pencil outline by edward burne jo...
97,"Every plant in the grass is set formally, grow...",EVERY PLANT IN THE GRASS IS SET FORMALLY GROWS...,every plant in the grass is set formally grows...,every plant in the grass is set formally grows...
98,Exquisite order and universal with eternal lif...,EXQUISITE ORDER AND UNIVERSAL WITH ETERNAL LIF...,exquisite order and universal with eternal lif...,exquisite order and universal with eternal lif...


In [42]:
wer = jiwer.wer(list(data["reference_clean"]), list(data["hypothesis_clean"]))
print("\n".join(data["reference_clean"]))
print(f"WER: {wer * 100:.2f} %")

he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
stuff it into you his belly counseled him
after early nightfall the yellow lamps would light up here and there the squalid quarter of the brothels
hello bertie any good in your mind
number 10 fresh nelly is waiting on you good night husband
the music came nearer and he recalled the words the words of shelley is fragment upon the moon wandering companionless pale for weariness
the dull light fell more faintly upon the page whereon another equation began to unfold itself slowly and to spread abroad its widening tail
a cold lucid indifference reigned in his soul
the chaos in which his ardor extinguished itself was a cold indifferent knowledge of himself
at most by an alms given to a beggar whose blessing he fled from he might hope wearily to win for himself some measure of actual grace
well now ennis i declare you have a head and so